In [ ]:
# train_detector.py (can also be run as a notebook)
"""
Fine‑tune YOLOv8‑m on the training subset.
Usage: python train_detector.py
"""

import os
import numpy as np
import tensorflow as tf
import keras_cv
from PIL import Image
import xml.etree.ElementTree as ET

# Configuration
TRAIN_DIR = "data/train_subset"
BATCH_SIZE = 4
EPOCHS = 10
IMG_SIZE = 640

# -------------------------------
# 1. Load images and annotations
# -------------------------------
def parse_xml(xml_path):
    """Return list of bboxes and classes for one image."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    classes = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        # Map class names to IDs – simplified example
        cls_map = {"motorcycle": 0, "car": 1, "rider": 2, "truck": 3, "pedestrian": 4}
        cls_id = cls_map.get(name.lower(), -1)
        if cls_id >= 0:
            boxes.append([xmin, ymin, xmax, ymax])
            classes.append(cls_id)
    return boxes, classes

def load_dataset(data_dir):
    image_dir = os.path.join(data_dir, "images")
    label_dir = os.path.join(data_dir, "labels")
    images = []
    all_boxes = []
    all_classes = []
    for img_name in os.listdir(image_dir):
        img_path = os.path.join(image_dir, img_name)
        xml_path = os.path.join(label_dir, img_name.rsplit(".", 1)[0] + ".xml")
        if not os.path.exists(xml_path):
            continue
        img = Image.open(img_path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE))
        images.append(np.array(img, dtype=np.float32) / 255.0)
        boxes, classes = parse_xml(xml_path)
        # Normalize boxes to [0,1]
        boxes = np.array(boxes, dtype=np.float32)
        boxes[:, [0,2]] /= IMG_SIZE
        boxes[:, [1,3]] /= IMG_SIZE
        all_boxes.append(boxes)
        all_classes.append(classes)
    return np.array(images), all_boxes, all_classes

# -------------------------------
# 2. Build model
# -------------------------------
backbone = keras_cv.models.YOLOV8Backbone.from_preset("yolo_v8_m_backbone_coco")
model = keras_cv.models.YOLOV8Detector(
    num_classes=5,
    bounding_box_format="xyxy",
    backbone=backbone,
    fpn_depth=1,
)

# Freeze backbone for first few epochs
model.backbone.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4))

# -------------------------------
# 3. Training loop (simplified)
# -------------------------------
# In a real training, you'd use a data generator.
# This snippet demonstrates the flow; adapt to a full training loop if you have time.
print("Training script ready. Replace with full training loop or run in Colab.")
# Save dummy weights as placeholder
model.save("models/detector_trained.h5")
print("Model saved to models/detector_trained.h5")